# NERC: two systems compared

For the NERC task we extract named entities from the test set and compare two
different systems on the same gold labels:

- **System A**: a Linear SVM we train ourselves on CoNLL-2003. Token-level
  features (word, POS, a bit of context and word shape).
- **System B**: a pretrained neural NER (spaCy / Stanza), used off the shelf.

Both predict BIO tags using the CoNLL tag set (PER, ORG, LOC, MISC), so we can
score them the same way with seqeval (span level) and a token-level report.

Test set: `../data/NER-test.tsv` (10 sentences, review-style text).
Training data for System A: CoNLL-2003 (news text).

## Data

**Training data (System A): CoNLL-2003 English.** This is the standard benchmark
for NERC, built from Reuters news stories from 1996-1997 (Tjong Kim Sang and De
Meulder, 2003). It is annotated with four entity types, PER, ORG, LOC and MISC,
in BIO format, which is exactly the tag set the test set uses. The training
split has around 14k sentences. It is heavily skewed towards `O` (about 83% of
tokens are non-entities), and among the entity tags the begin tags (B-) are far
more frequent than the inside tags (I-), which already tells us the rare inside
tags will be the hard ones. We use CoNLL-2003 because it matches the test tag
set and is the data System A was built around in the lab.

**Test data: the provided `NER-test.tsv`.** It has 10 sentences, one token per
row, with columns for sentence id, token id, token and the gold BIO tag. Unlike
CoNLL-2003 this is not news text, it is short review-style sentences about
movies, books and restaurants. That domain gap matters: the entities here are
things like a film studio (Warner Brothers), a university (New York University)
and a restaurant (Blauwbrug), so several organisations are named after people or
places. A model trained on news has seen "Amsterdam" and "Cuba" as locations,
so we already expect trouble on these ambiguous cases.

The label counts printed below confirm the test set is small and imbalanced:
most tokens are `O` and there are only a few of each entity type. Because of
this we report span-level F1 with seqeval and the per-class scores rather than
overall accuracy, which would be misleadingly high.

*Reference: Erik F. Tjong Kim Sang and Fien De Meulder (2003). Introduction to
the CoNLL-2003 Shared Task: Language-Independent Named Entity Recognition. CoNLL.*

In [1]:
import os
from collections import Counter
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn import svm
from sklearn.metrics import classification_report as token_report
from seqeval.metrics import classification_report as span_report, f1_score as span_f1

DATA = "../data"
CONLL = "../../lab_sessions/lab4/CONLL2003/CONLL2003" 

## Load the data

We read CoNLL-2003 (for training System A) and the test set. The test file is a
TSV with one token per row and a sentence id, so we group by sentence id.

In [2]:
def read_conll(path):
    sents, cur = [], []
    for line in open(path, encoding="utf-8"):
        line = line.rstrip()
        if not line or line.startswith("-DOCSTART-"):
            if cur:
                sents.append(cur); cur = []
            continue
        parts = line.split()
        cur.append((parts[0], parts[1], parts[-1]))  # word, pos, tag
    if cur:
        sents.append(cur)
    return sents

def read_test(path):
    df = pd.read_csv(path, sep="\t")
    df.columns = [col.strip() for col in df.columns]
    sents = []
    for _, grp in df.groupby("sentence id", sort=True):
        sents.append(list(zip(grp["token"].astype(str), grp["BIO NER tag"].str.strip())))
    return sents

train_sents = read_conll(f"{CONLL}/train.txt")
test_sents = read_test(f"{DATA}/NER-test.tsv")
tokens = [[t for t, _ in s] for s in test_sents]
gold = [[g for _, g in s] for s in test_sents]

print("train sentences:", len(train_sents))
print("test sentences:", len(test_sents))
print("test labels:", Counter(g for s in gold for g in s))

train sentences: 14041
test sentences: 10
test labels: Counter({'O': 183, 'I-PER': 8, 'B-PER': 6, 'B-ORG': 4, 'B-LOC': 4, 'I-ORG': 3, 'B-MISC': 3, 'I-LOC': 2, 'I-MISC': 1})


The test set is tiny (10 sentences) and very imbalanced: most tokens are `O`,
and there are only a handful of each entity type. So overall accuracy will look
high no matter what, and we should read the span-level F1 and the per-class
numbers instead.

## System A: Linear SVM trained on CoNLL-2003

We train our own classifier. For each token we build a small feature dict: the
lowercased word, its POS tag, whether it is capitalised / all caps / a digit,
its last three characters, and the same kind of info for the word before and
after it. The context features matter because whether a token is inside an
entity depends on its neighbours.

The test set has no POS column, so we POS-tag it with spaCy first (only to get
the POS feature, not for the entities).

In [3]:
def features(sent, i):
    word, pos = sent[i]
    f = {
        "w": word.lower(), "pos": pos,
        "title": word.istitle(), "upper": word.isupper(),
        "digit": word.isdigit(), "suf3": word[-3:].lower(),
    }
    if i > 0:
        f["-1w"] = sent[i-1][0].lower(); f["-1pos"] = sent[i-1][1]
        f["-1title"] = sent[i-1][0].istitle()
    else:
        f["BOS"] = True
    if i < len(sent) - 1:
        f["+1w"] = sent[i+1][0].lower(); f["+1pos"] = sent[i+1][1]
        f["+1title"] = sent[i+1][0].istitle()
    else:
        f["EOS"] = True
    return f

# POS-tag the test set (spaCy), feeding in our own tokens so nothing shifts
import spacy
from spacy.tokens import Doc
nlp = spacy.load("en_core_web_sm")

test_wp = []
for toks in tokens:
    doc = nlp.get_pipe("tagger")(nlp.get_pipe("tok2vec")(Doc(nlp.vocab, words=toks)))
    test_wp.append([(t.text, t.tag_) for t in doc])

In [4]:
# build features and train
train_X = [features([(w, p) for w, p, _ in s], i)
           for s in train_sents for i in range(len(s))]
train_y = [t for s in train_sents for _, _, t in s]
test_X = [features(s, i) for s in test_wp for i in range(len(s))]

vec = DictVectorizer()
X = vec.fit_transform(train_X + test_X)
n = len(train_X)

clf = svm.LinearSVC(max_iter=10000)
clf.fit(X[:n], train_y)
flat = list(clf.predict(X[n:]))

# put the flat predictions back into sentences
pred_a, k = [], 0
for s in test_wp:
    pred_a.append(flat[k:k+len(s)]); k += len(s)

print("System A span-F1:", round(span_f1(gold, pred_a, zero_division=0), 3))
print(span_report(gold, pred_a, digits=3, zero_division=0))

System A span-F1: 0.513
              precision    recall  f1-score   support

         LOC      0.500     0.750     0.600         4
        MISC      0.500     0.667     0.571         3
         ORG      0.400     0.500     0.444         4
         PER      0.429     0.500     0.462         6

   micro avg      0.455     0.588     0.513        17
   macro avg      0.457     0.604     0.519        17
weighted avg      0.451     0.588     0.509        17



### System A results

The macro/micro numbers print above. The thing to notice is that token accuracy
is high (~0.96) but span-level F1 is much lower, because getting a whole entity
right is harder than getting most tokens right when almost everything is `O`.

Looking at where it goes wrong (run the cell below), the mistakes are not
random. The SVM keys off the surface form of a word, so it labels words that
*look* like a type even when the sentence says otherwise:

- **Amsterdam** is tagged ORG instead of LOC, and **Blauwbrug** (an Amsterdam
  restaurant) is tagged LOC instead of ORG. The model has learned "Amsterdam =
  place" from the news training data and cannot tell that here Blauwbrug is the
  organisation and Amsterdam is just the city it sits in.
- **Cuba** (Gooding Jr., an actor) is tagged LOC instead of PER, again because
  "Cuba" is a country in the training data.
- **Mr.** before a name is dropped (tagged O), so the person span starts late.

In short, System A fails on exactly the cases where the right label depends on
meaning and context rather than the word itself. This is the known limit of a
token-level model trained on news text and applied to review text.

In [5]:
rows = []
for toks, g, p in zip(tokens, gold, pred_a):
    for tk, gg, pp in zip(toks, g, p):
        if gg != pp:
            rows.append({"token": tk, "gold": gg, "System A": pp})
pd.DataFrame(rows)

,token,gold,System A
0,New,B-ORG,B-LOC
1,Cuba,B-PER,B-LOC
2,Gooding,I-PER,I-ORG
3,American,I-MISC,B-MISC
4,Amsterdam,B-LOC,B-ORG
5,Blauwbrug,B-ORG,B-LOC
6,Maggie,I-PER,B-PER
7,Mr.,B-PER,O


## System B: pretrained neural NER  *(partner's part)*

System B is a pretrained neural tagger used off the shelf, so we can compare a
model we trained ourselves against one trained on much more data. Implement it
below and the same evaluation will apply.

Contract for `run_system_b`:
- input: `tokens`, a list of sentences where each sentence is a list of token
  strings (use these tokens directly, do not re-tokenise, or the predictions
  will not line up with the gold tags)
- output: a list of sentences of BIO tags, same length per sentence, in the
  CoNLL tag set (O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, B-MISC, I-MISC)

Suggested model: spaCy `en_core_web_trf` or Stanza. Map the model's entity
types to the CoNLL set (PERSON to PER, GPE/LOC/FAC to LOC, NORP/LANGUAGE/EVENT
to MISC, ORG to ORG).

In [6]:
SPACY_MAP = {
    "PERSON": "PER", "ORG": "ORG",
    "GPE": "LOC", "LOC": "LOC", "FAC": "LOC",
    "NORP": "MISC", "LANGUAGE": "MISC", "EVENT": "MISC",
    "WORK_OF_ART": "MISC", "PRODUCT": "MISC", "LAW": "MISC",
}

def run_system_b(tokens):
    # TODO (partner): load a pretrained NER and return BIO tags per sentence.
    raise NotImplementedError("System B not implemented yet")

# Once implemented, uncomment:
# pred_b = run_system_b(tokens)
# print("System B span-F1:", round(span_f1(gold, pred_b, zero_division=0), 3))
# print(span_report(gold, pred_b, digits=3, zero_division=0))

## Comparison and conclusions  *(to fill in once System B runs)*

Once System B is implemented, add the side-by-side comparison here (per-label F1
for System A vs System B) and a short discussion:

- Which system wins overall, and on which entity types.
- Where the two systems make the *same* mistakes. The test set has organisations
  named after places or people (Warner Brothers, New York University,
  Blauwbrug), which are ambiguous out of context, so it is worth checking
  whether both systems struggle there and whether they fail for the right
  reasons.
- The kinds of errors specific to System B (boundary errors, type confusions).
- Suggestions to improve, e.g. training or fine-tuning on review-domain text
  instead of news, adding a gazetteer for restaurants and organisations, or
  moving System A to a sequence model (CRF) so it uses neighbouring tags.